#Configuration



In [ ]:
import os
import pandas as pd
import requests
import json
import base64
import time
import glob

# DataForSEO Credentials
login = os.environ["DATAFORSEO_LOGIN"]
password = os.environ["DATAFORSEO_PASSWORD"]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RA/health_care/dentist_all_keywords"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Prepare Keyword CSV File

In [ ]:
# Define the input directory path
input_dir = os.path.join(base_path, "input")
if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")


sample_data = {
    'location_name': ['roanoke_lynchburg', 'blacksburg', 'christiansburg','DC','NY','Charlotte',None,None,None,None,None],
    'location_code': [200573, 1027041, 1027077,200511,21167,200517,None,None,None,None,None],
    # 'keyword1_dentist': ['dentist','orthodontist','endodontist','periodontist','oral surgeon','prosthodontist']
    'keyword1_dentist': ['dentist', 'pediatric dentist', 'dental clinic','general dentist','family dentist','orthodontist','endodontist','periodontist','oral surgeon','emergency dentist','prosthodontist'],
    # 'keyword2_clinic': ['clinic', 'walk-in clinic', 'community health clinic'],
    # 'keyword3_hospital': ['general hospital', 'specialty hospital',None],
    # 'keyword1_ur': ['urgent care', 'emergency room',None]
}
sample_df = pd.DataFrame(sample_data)
csv_path = os.path.join(input_dir, "keyword_dentist.csv") # Save to input directory
sample_df.to_csv(csv_path, index=False)

print(f"Sample file saved to: {csv_path}")

from IPython.display import display
display(sample_df)

# POST Tasks to DataForSEO
## Wait 20min after running this cell

In [ ]:
def post_dataforseo_task(post_url, location_code, keyword, depth=100):
    post_payload = [{
        "location_code": location_code,
        "language_code": "en",
        "keyword": keyword,
        "depth": depth
    }]

    cred_string = f"{login}:{password}"
    cred_base64 = base64.b64encode(cred_string.encode("utf-8")).decode("utf-8")
    headers = {
        'Authorization': f'Basic {cred_base64}',
        'Content-Type': 'application/json'
    }

    print(f"  POST task: Location Code={location_code}, Keyword='{keyword}'")

    try:
        response = requests.post(post_url, headers=headers, json=post_payload)
        response.raise_for_status() # Check for HTTP status code errors
        result = response.json()

        if result and result.get("tasks") and result["tasks"][0].get("id"):
            task_id = result["tasks"][0]["id"]
            print(f"  Task POST successfully for keyword: '{keyword}'. Task ID: {task_id}")
            return task_id
        else:
            print(f"  Task POST successful, but no Task ID found in response. Keyword: '{keyword}'. Response: {result}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"  Error POST task for keyword: '{keyword}': {e}")
        return None
    except json.JSONDecodeError:
        print(f"  Task POST successful, but could not parse JSON response. Keyword: '{keyword}'. Response text: {response.text}")
        return None




In [ ]:
print("--- Starting Task POST ---")
input_dir = os.path.join(base_path, "input")
temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")

if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)
    print(f"Created directory: {temp_dir}")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")


csv_path = os.path.join(input_dir, "keyword_dentist.csv")
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"Error: '{csv_path}' not found. Please run the previous cell first.")
    tasks_to_submit_df = pd.DataFrame()
else:
    tasks_to_submit_df = df.copy()

task_list_csv_path = os.path.join(temp_dir, "task_list.csv")

# Load already posted tasks to avoid re-posting
existing_tasks = set()
try:
    existing_tasks_df = pd.read_csv(task_list_csv_path)
    # Create a set of (location_name, keyword, api_type) tuples for efficient lookup
    for index, row in existing_tasks_df.iterrows():
        existing_tasks.add((row['location_name'], row['keyword'], row['api_type']))
    print(f"Loaded {len(existing_tasks)} existing tasks to skip.")
except FileNotFoundError:
    print("No existing task file found. Will post all tasks.")


write_header = not os.path.exists(task_list_csv_path)

api_endpoints = {
    "local_finder": "https://api.dataforseo.com/v3/serp/google/local_finder/task_post",
    "maps": "https://api.dataforseo.com/v3/serp/google/maps/task_post"
}

if not tasks_to_submit_df.empty:
    keyword_columns = [col for col in tasks_to_submit_df.columns if col.startswith('keyword')]

    if not keyword_columns:
        print("Error: No keyword columns found.")
    else:
        for keyword_col in keyword_columns:
            print(f"Processing keyword column: '{keyword_col}'")
            keywords_in_column = tasks_to_submit_df[keyword_col].dropna().tolist()

            if not keywords_in_column:
                print(f"  Keyword column '{keyword_col}' has no keywords. Skipping.")
                continue

            from itertools import combinations
            keyword_combinations = []
            for i in range(1, min(len(keywords_in_column), 3) + 1):
            #for i in range(1, min(len(keywords_in_column), 1) + 1):
                 keyword_combinations.extend(list(combinations(keywords_in_column, i)))

            if not keyword_combinations:
                 print(f"  Could not generate combinations for keyword column '{keyword_col}'. Skipping.")
                 continue

            print(f"  Keyword combinations for column '{keyword_col}': {keyword_combinations}")

            for location_index, location_row in tasks_to_submit_df.iterrows():
                location_name = location_row['location_name']
                location_code_val = location_row['location_code']

                if pd.isna(location_code_val):
                    print(f"  Skipping task POST for location '{location_name}' due to missing location_code.")
                    continue

                location_code = int(location_code_val)
                print(f"  Processing tasks for location '{location_name}' ({location_code}):")

                for combo in keyword_combinations:
                    combined_keyword = "+".join(combo)

                    for api_name, api_url in api_endpoints.items():
                        # Skip posted task
                        if (location_name, combined_keyword, api_name) in existing_tasks:
                            print(f"    Skipping already posted task: Keyword='{combined_keyword}', API='{api_name}'")
                            continue # Move to the next api_name

                        print(f"    Combined keyword: '{combined_keyword}'")
                        raw_output_dir = os.path.join(temp_dir, api_name)
                        if not os.path.exists(raw_output_dir):
                            os.makedirs(raw_output_dir)
                            print(f"Created directory: {raw_output_dir}")

                        task_id = post_dataforseo_task(api_url, location_code, combined_keyword)
                        if task_id:
                            safe_combined_keyword = combined_keyword.replace(' ', '_').replace('/', '_').replace('\\\\', '_')
                            raw_json_filename = f"{location_name}_{safe_combined_keyword}_{api_name}.json"
                            raw_json_path = os.path.join(raw_output_dir, raw_json_filename)

                            current_task_df = pd.DataFrame([{
                                "task_id": task_id,
                                "api_type": api_name,
                                "location_name": location_name,
                                "keyword": combined_keyword,
                                "raw_json_path": raw_json_path
                            }])

                            current_task_df.to_csv(task_list_csv_path, mode='a', header=write_header, index=False)
                            write_header = False
                            print(f"  Task info appended to: '{task_list_csv_path}'")

                        time.sleep(1)


    print(f"All new tasks posted and incrementally saved to '{task_list_csv_path}').")
    print("!!! IMPORTANT: Please wait 20 minutes before running the next cell to allow results to become available. !!!")
else:
    print("Keyword database file not found or empty.")

#Get Task Results
## Wait 20 mins after running previous cell

In [ ]:
def get_dataforseo_results(task_id, api_type, raw_json_path):
    if api_type == "local_finder":
        get_url_template = "https://api.dataforseo.com/v3/serp/google/local_finder/task_get/advanced/{}"
    elif api_type == "maps":
        get_url_template = "https://api.dataforseo.com/v3/serp/google/maps/task_get/advanced/{}"
    else:
        print(f"  Unknown API type: {api_type}")
        return

    get_url = get_url_template.format(task_id)

    cred_string = f"{login}:{password}"
    cred_base64 = base64.b64encode(cred_string.encode("utf-8")).decode("utf-8")
    headers = {'Authorization': f'Basic {cred_base64}'}

    try:
        response = requests.get(get_url, headers=headers)
        response.raise_for_status()
        result_data = response.json()

        raw_data_dir = os.path.dirname(raw_json_path)
        if not os.path.exists(raw_data_dir):
            os.makedirs(raw_data_dir)
            print(f"Created directory: {raw_data_dir}")

        with open(raw_json_path, "w", encoding="utf-8") as f:
            json.dump(result_data, f, indent=4, ensure_ascii=False)
        print(f"  Saved JSON results for Task ID {task_id} to {os.path.basename(raw_json_path)}") # Updated message
    except requests.exceptions.RequestException as e:
        print(f"  Error getting results for Task ID {task_id}: {e}")

In [ ]:
print("--- Get Task Results ---")
temp_dir = os.path.join(base_path, "temp")
task_list_path = os.path.join(temp_dir, "task_list.csv")
tasks_to_get = []
try:
    # Read the task list from CSV
    task_list_df = pd.read_csv(task_list_path)
    # Convert df rows to a list of dictionaries
    tasks_to_get = task_list_df.to_dict('records')
except FileNotFoundError:
    print(f"Error: '{task_list_path}' not found. Please run the task submission cell first.")

if tasks_to_get:
    print(f"Found {len(tasks_to_get)} tasks to get results for.")
    for task in tasks_to_get:

        # Skip result file already exists
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping: Result file already exists -> '{os.path.basename(task['raw_json_path'])}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll result retrieval attempts completed.")

# Process Data

In [ ]:
def parse_local_finder_results(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Could not read or parse {os.path.basename(file_path)}: {e}")
        return []

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            # Local Finder
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}

            extracted_data.append({
                "title": item.get("title"),
                "description": item.get("description"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "type": item.get("type")
            })
    return extracted_data

def parse_maps_results(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Could not read or parse {os.path.basename(file_path)}: {e}")
        return []

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            # Maps
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}
            rating_distribution = item.get("rating_distribution", {})
            if not isinstance(rating_distribution, dict): rating_distribution = {}
            address_info = item.get("address_info", {})
            if not isinstance(address_info, dict): address_info = {}

            extracted_data.append({
                "title": item.get("title"),
                "address": item.get("address"),
                "latitude": item.get("latitude"),
                "longitude": item.get("longitude"),
                "zip": address_info.get("zip"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "rating_1_star": rating_distribution.get("1", 0),
                "rating_2_star": rating_distribution.get("2", 0),
                "rating_3_star": rating_distribution.get("3", 0),
                "rating_4_star": rating_distribution.get("4", 0),
                "rating_5_star": rating_distribution.get("5", 0),
                "type": item.get("type")
            })
    return extracted_data


In [ ]:
print("--- Data Processing ---")

temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")

local_finder_raw_json_dir = os.path.join(temp_dir, "local_finder")
maps_raw_json_dir = os.path.join(temp_dir, "maps")

local_finder_output_raw_dir = os.path.join(output_dir, "local_finder", "raw") # before deduplication
local_finder_output_processed_dir = os.path.join(output_dir, "local_finder", "processed") # processed data
maps_output_raw_dir = os.path.join(output_dir, "maps", "raw") # before deduplication
maps_output_processed_dir = os.path.join(output_dir, "maps", "processed") # processed data

for d in [local_finder_output_raw_dir, local_finder_output_processed_dir, maps_output_raw_dir, maps_output_processed_dir]:
    if not os.path.exists(d):
        os.makedirs(d)
        print(f"Created directory: {d}")


local_finder_json_files = glob.glob(os.path.join(local_finder_raw_json_dir, "*.json"))
maps_json_files = glob.glob(os.path.join(maps_raw_json_dir, "*.json"))

local_finder_data = []
maps_data = []

if not local_finder_json_files and not maps_json_files:
    print("No JSON result files found for processing in temp/local_finder or temp/maps.")
else:
    print(f"Found {len(local_finder_json_files)} Local Finder files and {len(maps_json_files)} Maps files to process.") # Updated message

    # Process Local Finder files
    for file in local_finder_json_files:
        print(f"Processing Local Finder file: {os.path.basename(file)}")
        local_finder_data.extend(parse_local_finder_results(file))

    # Process Maps files
    for file in maps_json_files:
        print(f"Processing Maps file: {os.path.basename(file)}")
        maps_data.extend(parse_maps_results(file))


    # Process Local Finder data
    if local_finder_data:
        local_finder_df = pd.DataFrame(local_finder_data)
        print(f"\nLocal Finder locations before deduplication: {len(local_finder_df)}")

        # Save processed data before deduplication to output/local_finder/raw
        local_finder_raw_csv_path = os.path.join(local_finder_output_raw_dir, "local_finder_raw_summary.csv")
        local_finder_df.to_csv(local_finder_raw_csv_path, index=False, encoding='utf-8-sig')
        print(f"Successfully saved Local Finder data BEFORE deduplication to: '{local_finder_raw_csv_path}'")


        # Deduplicate based on title
        local_finder_deduplicated_df = local_finder_df.drop_duplicates(subset=['title'], keep='first')
        print(f"Local Finder locations after deduplication: {len(local_finder_deduplicated_df)}")

        # Sort Local Finder data
        local_finder_sorted_df = local_finder_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        # Save processed data after deduplication to output/local_finder/processed
        local_finder_processed_csv_path = os.path.join(local_finder_output_processed_dir, "local_finder_processed_summary.csv")
        local_finder_sorted_df.to_csv(local_finder_processed_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved deduplicated Local Finder data to: '{local_finder_processed_csv_path}'")

        print("\nFinal Local Finder data preview:")
        display(local_finder_sorted_df.head(10))
    else:
        print("\nNo data extracted from Local Finder files.")

    # Process Maps data
    if maps_data:
        maps_df = pd.DataFrame(maps_data)
        print(f"\nMaps locations before deduplication: {len(maps_df)}")

        # Save processed data before deduplication to output/maps/raw
        maps_raw_csv_path = os.path.join(maps_output_raw_dir, "maps_raw_summary.csv")
        maps_df.to_csv(maps_raw_csv_path, index=False, encoding='utf-8-sig')
        print(f"Successfully saved Maps data BEFORE deduplication to: '{maps_raw_csv_path}'")

         # Deduplicate based on title and address
        maps_deduplicated_df = maps_df.drop_duplicates(subset=['title', 'address'], keep='first')
        print(f"Maps locations after deduplication: {len(maps_deduplicated_df)}")
        # Sort Maps data
        maps_sorted_df = maps_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        # Save processed data after deduplication to output/maps/processed
        maps_processed_csv_path = os.path.join(maps_output_processed_dir, "maps_processed_summary.csv")
        maps_sorted_df.to_csv(maps_processed_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSuccessfully saved deduplicated Maps data to: '{maps_processed_csv_path}'")


        print("\nFinal Maps data preview:")
        display(maps_sorted_df.head(10))
    else:
         print("\nNo data extracted from Maps files.")


    print("\nAll result processing attempts completed.")

# Merge and Deduplicate Processed Results

In [ ]:
print("--- Merging Processed Results ---")

output_dir = os.path.join(base_path, "output")
local_finder_processed_path = os.path.join(output_dir, "local_finder", "processed", "local_finder_processed_summary.csv")
maps_processed_path = os.path.join(output_dir, "maps", "processed", "maps_processed_summary.csv")

local_finder_processed_df = pd.DataFrame()
maps_processed_df = pd.DataFrame()

try:
    local_finder_processed_df = pd.read_csv(local_finder_processed_path)
    print(f"Loaded Local Finder processed data from: '{local_finder_processed_path}'")
except FileNotFoundError:
    print(f"Error: '{local_finder_processed_path}' not found.")

try:
    maps_processed_df = pd.read_csv(maps_processed_path)
    print(f"Loaded Maps processed data from: '{maps_processed_path}'")
except FileNotFoundError:
    print(f"Error: '{maps_processed_path}' not found.")

# Check if both df were loaded successfully
if not local_finder_processed_df.empty and not maps_processed_df.empty:


    # Merge the two dataframes based on 'title'
    merged_df = pd.merge(local_finder_processed_df, maps_processed_df, on='title', how='outer', suffixes=('_local_finder', '_maps'))

    print(f"Merged df shape: {merged_df.shape}")

    # Create indicator columns based is in which or both api
    merged_df['isFinder'] = merged_df['rating_value_local_finder'].apply(lambda x: 0 if pd.isna(x) else 1)
    merged_df['isMap'] = merged_df['rating_value_maps'].apply(lambda x: 0 if pd.isna(x) else 1)
    merged_df['isBoth'] = merged_df.apply(lambda row: 1 if row['isFinder'] == 1 and row['isMap'] == 1 else 0, axis=1)

    final_columns = {
        'title': 'title',
        'votes_count': 'votes_count_local_finder',
        'rating_value': 'rating_value_local_finder',
        'address': 'address',
        'latitude': 'latitude',
        'longitude': 'longitude',
        'zip': 'zip',
        'rating_1_star': 'rating_1_star',
        'rating_2_star': 'rating_2_star',
        'rating_3_star': 'rating_3_star',
        'rating_4_star': 'rating_4_star',
        'rating_5_star': 'rating_5_star',
        'isFinder': 'isFinder',
        'isMap': 'isMap',
        'isBoth': 'isBoth'
    }

    merged_df['votes_count'] = merged_df['votes_count_local_finder'].fillna(merged_df['votes_count_maps'])
    merged_df['rating_value'] = merged_df['rating_value_local_finder'].fillna(merged_df['rating_value_maps'])
    merged_df['address'] = merged_df['address'].fillna(merged_df['address'])
    merged_df['latitude'] = merged_df['latitude'].fillna(merged_df['latitude'])
    merged_df['longitude'] = merged_df['longitude'].fillna(merged_df['longitude'])
    merged_df['zip'] = merged_df['zip'].fillna(merged_df['zip'])


    for i in range(1, 6):
        star_col = f'rating_{i}_star'
        if f'{star_col}_maps' in merged_df.columns:
             merged_df[star_col] = merged_df[f'{star_col}_maps'].fillna(0)
        else:
             merged_df[star_col] = merged_df[star_col].fillna(0)


    final_df = merged_df[['title', 'votes_count', 'rating_value', 'address', 'latitude', 'longitude','zip', 'rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star', 'isFinder', 'isMap', 'isBoth']].copy() # Include new columns

    print(f"\nCombined df shape before deduplication: {final_df.shape}")

    # Deduplicate the combined df based on 'title'
    final_deduplicated_df = final_df.drop_duplicates(subset=['title'], keep='first')

    print(f"Combined df shape after deduplication: {final_deduplicated_df.shape}")

    # Sort the final df by rating_value and votes_count
    final_sorted_df = final_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

    print("\nFinal Data Preview:")
    display(final_sorted_df.head(10))


    # save
    final_output_dir = os.path.join(output_dir, "merged_deduplicated")
    if not os.path.exists(final_output_dir):
        os.makedirs(final_output_dir)
        print(f"Created directory: {final_output_dir}")

    final_csv_path = os.path.join(final_output_dir, "merged_deduplicated_summary.csv")
    final_sorted_df.to_csv(final_csv_path, index=False, encoding='utf-8-sig') # Save sorted dataframe
    print(f"\nSaved final data to: '{final_csv_path}'")


elif local_finder_processed_df.empty:
    print("\nCannot merge. Local Finder processed data not found.")
elif maps_processed_df.empty:
     print("\nCannot merge. Maps processed data not found.")

# Get isFinder==1 but isMap==0 as new keyword and POST task

In [ ]:
print("--- Posting Supplementary Tasks ---")

output_dir = os.path.join(base_path, "output")
input_dir = os.path.join(base_path, "input")
temp_dir = os.path.join(base_path, "temp")

supplement_task_list_csv_path = os.path.join(temp_dir, "supplement_task_list.csv")


existing_tasks = set()
try:
    existing_tasks_df = pd.read_csv(supplement_task_list_csv_path)
    for index, row in existing_tasks_df.iterrows():
        existing_tasks.add((row['location_name'], row['keyword']))
    print(f"Loaded {len(existing_tasks)} existing supplementary tasks to skip.")
except FileNotFoundError:
    print("No existing supplementary task file found. Will post all tasks.")


merged_deduplicated_path = os.path.join(output_dir, "merged_deduplicated", "merged_deduplicated_summary.csv")
try:
    merged_df = pd.read_csv(merged_deduplicated_path)
    print(f"Loaded merged data from: '{merged_deduplicated_path}'")
except FileNotFoundError:
    print(f"Error: '{merged_deduplicated_path}' not found. Please run the previous cells first.")
    merged_df = pd.DataFrame()

if not merged_df.empty:
    supplement_keywords_df = merged_df[(merged_df['isFinder'] == 1) & (merged_df['isMap'] == 0)]
    supplement_keywords = supplement_keywords_df['title'].dropna().unique().tolist()

    print(f"Found {len(supplement_keywords)} keywords to supplement from Maps API.")

    if supplement_keywords:
        keyword_db_path = os.path.join(input_dir, "keyword_dentist.csv")
        try:
            locations_df = pd.read_csv(keyword_db_path)
        except FileNotFoundError:
            print(f"Error: Original keyword database '{keyword_db_path}' not found.")
            locations_df = pd.DataFrame()

        if not locations_df.empty:
            write_header = not os.path.exists(supplement_task_list_csv_path)
            maps_api_url = "https://api.dataforseo.com/v3/serp/google/maps/task_post"

            supplement_maps_output_dir = os.path.join(temp_dir, "maps_supplement")
            if not os.path.exists(supplement_maps_output_dir):
                os.makedirs(supplement_maps_output_dir)
                print(f"Created directory: {supplement_maps_output_dir}")

            for _, location_row in locations_df.iterrows():
                if pd.isna(location_row['location_code']):
                    print(f"Skip NAN location code")
                    continue

                location_name = location_row['location_name']
                location_code = int(location_row['location_code'])

                print(f"\nProcessing supplementary tasks for location '{location_name}' ({location_code}):")
                for keyword in supplement_keywords:
                    # check if existing task, if so, skip
                    if (location_name, keyword) in existing_tasks:
                        print(f"  Skipping already posted task for keyword: '{keyword}'")
                        continue

                    task_id = post_dataforseo_task(maps_api_url, location_code, keyword)

                    if task_id:
                        safe_keyword = keyword.replace(' ', '_').replace('/', '_').replace('\\\\', '_')
                        raw_json_filename = f"{location_name}_{safe_keyword}_maps.json"
                        raw_json_path = os.path.join(supplement_maps_output_dir, raw_json_filename)

                        current_task_df = pd.DataFrame([{
                            "task_id": task_id,
                            "api_type": "maps",
                            "location_name": location_name,
                            "keyword": keyword,
                            "raw_json_path": raw_json_path
                        }])

                        current_task_df.to_csv(supplement_task_list_csv_path, mode='a', header=write_header, index=False)

                        write_header = False

                        print(f"  Task info for '{keyword}' appended to: '{supplement_task_list_csv_path}'")

                    time.sleep(1)

            print(f"All supplementary tasks posted and incrementally saved.")
            print("!!! IMPORTANT: Please wait 20 minutes before running the next cell. !!!")
else:
    print("\nMerged data is empty. Cannot proceed.")



# Get supplement result

In [ ]:
print("--- Get & Process Supplementary Task Results ---")

temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")
supplement_task_list_path = os.path.join(temp_dir, "supplement_task_list.csv")

# Get Task Results
tasks_to_get = []
try:
    supplement_task_df = pd.read_csv(supplement_task_list_path)
    tasks_to_get = supplement_task_df.to_dict('records')
    print(f"Found {len(tasks_to_get)} supplementary tasks to get results for.")

    for task in tasks_to_get:
        # Skip existing files
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping: Result file already exists -> '{os.path.basename(task['raw_json_path'])}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll supplementary result retrieval attempts completed.")

except FileNotFoundError:
    print(f"Error: '{supplement_task_list_path}' not found. No supplementary tasks to process.")


# Process Data
if tasks_to_get:
    maps_supplement_raw_json_dir = os.path.join(temp_dir, "maps_supplement")
    supplement_json_files = glob.glob(os.path.join(maps_supplement_raw_json_dir, "*.json"))
    supplement_data = []

    print(f"\nFound {len(supplement_json_files)} supplementary JSON files to process.")

    for file in supplement_json_files:
        print(f"Processing supplementary file: {os.path.basename(file)}")
        supplement_data.extend(parse_maps_results(file))

    if supplement_data:
        supplement_df = pd.DataFrame(supplement_data)
        print(f"\nSupplementary locations before deduplication: {len(supplement_df)}")

        supplement_deduplicated_df = supplement_df.drop_duplicates(subset=['title', 'address'], keep='first')
        print(f"Supplementary locations after deduplication: {len(supplement_deduplicated_df)}")

        supplement_output_dir = os.path.join(output_dir, "supplement")
        if not os.path.exists(supplement_output_dir):
            os.makedirs(supplement_output_dir)

        supplement_csv_path = os.path.join(supplement_output_dir, "supplement_processed_summary.csv")
        supplement_deduplicated_df.to_csv(supplement_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved processed supplementary data to: '{supplement_csv_path}'")
        display(supplement_deduplicated_df.head())
    else:
        print("\nNo data extracted from supplementary Maps files.")
else:
    print("\nNo supplementary tasks were run, skipping processing.")


# Merge and process

In [ ]:
print("--- Final Merging and Consolidation ---")

output_dir = os.path.join(base_path, "output")

original_merged_path = os.path.join(output_dir, "merged_deduplicated", "merged_deduplicated_summary.csv")
supplement_path = os.path.join(output_dir, "supplement", "supplement_processed_summary.csv")

try:
    original_df = pd.read_csv(original_merged_path)
    print(f"Loaded original merged data: {original_df.shape}")
except FileNotFoundError:
    print(f"Error: '{original_merged_path}' not found.")
    original_df = pd.DataFrame()

try:
    supplement_df = pd.read_csv(supplement_path)
    print(f"Loaded supplementary data: {supplement_df.shape}")
except FileNotFoundError:
    print(f"'{supplement_path}' not found.")
    supplement_df = pd.DataFrame()


if not original_df.empty:
    if not supplement_df.empty:
        print("\\nPreparing supplementary data...")
        supplement_df['isFinder'] = 1
        supplement_df['isMap'] = 1
        supplement_df['isBoth'] = 1

        supplement_df = supplement_df.reindex(columns=original_df.columns)

        # Concatenate
        combined_df = pd.concat([original_df, supplement_df], ignore_index=True)
        print(f"\\nShape before final consolidation: {combined_df.shape}")

        agg_funcs = {
            # For all columns except 'title', take the first non-null value within each group
            col: 'first' for col in combined_df.columns if col != 'title'
        }
        # If any row has a 1, the result is 1
        agg_funcs['isFinder'] = 'max'
        agg_funcs['isMap'] = 'max'

        final_df = combined_df.groupby('title', as_index=False).agg(agg_funcs)

        # Recalculate the 'isBoth' flag for accuracy based on the final merged flags
        final_df['isBoth'] = ((final_df['isFinder'] == 1) & (final_df['isMap'] == 1)).astype(int)

        print(f"Shape after consolidation: {final_df.shape}")

    else:
        # If there is no supplementary data, the final result is the original data
        final_df = original_df
        print("\\nNo supplementary data to merge.")

    # Sort and save the final result
    final_sorted_df = final_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

    final_output_dir = os.path.join(output_dir, "final")
    os.makedirs(final_output_dir, exist_ok=True)

    final_csv_path = os.path.join(final_output_dir, "final_processed.csv")
    final_sorted_df.to_csv(final_csv_path, index=False, encoding='utf-8-sig')

    print(f"\\nSaved final processed data to: '{final_csv_path}'")
    print("\\nFinal Processed Data Preview:")
    display(final_sorted_df.head(20))

else:
    print("Original merged data not found. Cannot create final processed file.")

# Rate Distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_all_keywords/output/final/final_processed.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    # ZIP codes for each region
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA) ZIP
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    # map ZIP code to location
    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'

        if zip_code in blacksburg_zips:
            return 'Blacksburg'
        elif zip_code in christiansburg_zips:
            return 'Christiansburg'
        elif zip_code in roanoke_lynchburg_zips:
            return 'Roanoke/Lynchburg'
        elif zip_code in dc_zips:
            return 'DC'
        elif zip_code in ny_zips:
            return 'NY'
        elif zip_code in charlotte_zips:
            return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)

    # Filter out locations don't belong to any of the defined regions
    df_filtered = df[df['location_name'] != 'Unknown'].copy()


    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value'], inplace=True)

    print(f"After mapping ZIP codes, {len(df_filtered)} records remain for analysis.")


    # Boxplot
    plt.figure(figsize=(12, 7))
    sns.boxplot(x='location_name', y='rating_value', data=df_filtered)
    plt.title('Distribution of Ratings by zipcode')
    plt.xlabel('Region')
    plt.ylabel('Rating Value')
    plt.grid(True)
    plt.show()

    # Histograms * 5
    plt.figure(figsize=(14, 8))
    sns.histplot(data=df_filtered, x='rating_value', hue='location_name', multiple='dodge', shrink=.8, bins=15)
    plt.title('Histogram of Ratings by zipcode')
    plt.xlabel('Rating Value')
    plt.ylabel('Count')
    plt.grid(True)
    plt.show()

    # Violin Plot of Rating Value
    plt.figure(figsize=(12, 7))
    sns.violinplot(x='location_name', y='rating_value', data=df_filtered)
    plt.title('Distribution of Rating Value by zipcode')
    plt.xlabel('Region')
    plt.ylabel('Rating Value')
    plt.grid(True)
    plt.show()


    rating_summary = df_filtered.groupby('location_name')['rating_value'].describe()
    print("\nSummary Statistics for Rating Value by zipcode")
    print(rating_summary)

    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    star_cols_exist = all(col in df_filtered.columns for col in star_cols)

    if star_cols_exist:
        for col in star_cols:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)

        star_distribution = df_filtered.groupby('location_name')[star_cols].mean()
        print("\nStar Rating Distribution (Average Votes per Star per Region):")
        display(star_distribution)

        # Stacked Bar Chart
        star_distribution.plot(kind='bar', stacked=True, figsize=(14, 8))
        plt.title('Average Star Rating Distribution by ZIP Code-Defined Region')
        plt.xlabel('Region')
        plt.ylabel('Average Number of Votes')
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Star Rating')
        plt.tight_layout()
        plt.show()
    else:
        print("\nStar rating columns not found. Skipping star rating analysis.")

# NPI

In [ ]:
import pandas as pd
import os

nppes_data_path = "/content/drive/MyDrive/RA/health_care/NPI"
temp_output_path = "/content/drive/MyDrive/RA/health_care/dentist_all_keywords/temp"
main_npi_file = "npidata_pfile_20050523-20251012.csv"
practice_location_file = "pl_pfile_20050523-20251012.csv"

# ZIP codes for each region
blacksburg_zips = [24060, 24061, 24062, 24063]
christiansburg_zips = [24068, 24073]
roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA ZIPs
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
]

# Healthcare Provider Taxonomy Codes for dentist
dentist_taxonomy_codes = [
    '122300000X',  # Dentist
    '1223D0001X',  # Dental Public Health
    '1223E0200X',  # Endodontics
    '1223G0001X',  # General Practice
    '1223P0106X',  # Pediatric Dentistry
    '1223P0221X',  # Periodontics
    '1223P0300X',  # Prosthodontics
    '1223P0700X',  # Pediatric Dentistry (alternate code)
    '1223X0008X',  # Oral and Maxillofacial Surgery
    '1223X0400X',  # Orthodontics and Dentofacial Orthopedics
    '122400000X',  # Dental Assistant
    '124Q00000X',  # Dental Hygienist
    '126800000X'   # Dental Laboratory
]


main_columns_to_load = [
    'NPI', 'Entity Type Code',
    'Provider Organization Name (Legal Business Name)',
    'Provider Last Name (Legal Name)', 'Provider First Name',
    'Provider Business Practice Location Address Postal Code',
    'Provider First Line Business Practice Location Address',
    'Provider Second Line Business Practice Location Address'
] + [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]

location_columns_to_load = [
    'NPI', 'Provider Secondary Practice Location Address - Postal Code', 'Provider Secondary Practice Location Address- Address Line 1', 'Provider Secondary Practice Location Address-  Address Line 2'
]

dentist_chunks = []
print("Processing main NPI file in chunks...")
try:
    # Create an iterator to read the CSV in chunks of 100,000 rows
    chunk_iterator = pd.read_csv(
        os.path.join(nppes_data_path, main_npi_file),
        usecols=main_columns_to_load,
        chunksize=100000,
        low_memory=False,
        encoding='utf-8'
    )

    for i, chunk in enumerate(chunk_iterator):
        print(f"  - Processing chunk {i+1}...")
        taxonomy_cols = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]
        mask = chunk[taxonomy_cols].isin(dentist_taxonomy_codes).any(axis=1)
        dentists_in_chunk = chunk[mask]

        if not dentists_in_chunk.empty:
            dentist_chunks.append(dentists_in_chunk)

    if dentist_chunks:
        df_dentists = pd.concat(dentist_chunks, ignore_index=True)
        print(f"\nFound {len(df_dentists)} total dental-related records from all chunks.")
    else:
        df_dentists = pd.DataFrame()

except FileNotFoundError as e:
    print(f"Error loading main file: {e}")
    df_dentists = pd.DataFrame()


if not df_dentists.empty:
    dentist_npi_set = set(df_dentists['NPI'])

    # Process primary locations
    primary_locs = df_dentists[['NPI', 'Provider Business Practice Location Address Postal Code', 'Provider First Line Business Practice Location Address', 'Provider Second Line Business Practice Location Address']].copy()
    primary_locs.rename(columns={
        'Provider Business Practice Location Address Postal Code': 'zip',
        'Provider First Line Business Practice Location Address': 'address_1',
        'Provider Second Line Business Practice Location Address': 'address_2'
    }, inplace=True)

    # Process secondary locations
    all_locations = primary_locs
    try:
        print("Loading and filtering practice location file...")
        location_columns_to_load = ['NPI', 'Provider Secondary Practice Location Address - Postal Code', 'Provider Secondary Practice Location Address- Address Line 1', 'Provider Secondary Practice Location Address-  Address Line 2']
        df_locations = pd.read_csv(os.path.join(nppes_data_path, practice_location_file), usecols=location_columns_to_load, low_memory=False, encoding='utf-8')

        secondary_locs = df_locations[df_locations['NPI'].isin(dentist_npi_set)].copy()
        secondary_locs.rename(columns={
            'Provider Secondary Practice Location Address - Postal Code': 'zip',
            'Provider Secondary Practice Location Address- Address Line 1': 'address_1',
            'Provider Secondary Practice Location Address-  Address Line 2': 'address_2'
        }, inplace=True)

        all_locations = pd.concat([primary_locs, secondary_locs], ignore_index=True)
        print(f"Found {len(secondary_locs)} additional locations.")
    except FileNotFoundError:
        print("Practice location file not found, proceeding with primary locations only.")

    # Clean and merge location data
    all_locations['zip'] = all_locations['zip'].astype(str).str.split('-').str[0]
    all_locations.dropna(subset=['zip'], inplace=True)
    all_locations['zip'] = pd.to_numeric(all_locations['zip'], errors='coerce').astype('Int64')

    name_cols = ['NPI', 'Entity Type Code', 'Provider Organization Name (Legal Business Name)', 'Provider Last Name (Legal Name)', 'Provider First Name']
    df_dentists_with_locs = pd.merge(df_dentists[name_cols], all_locations, on='NPI').drop_duplicates()

    # Map to regions
    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        if zip_code in blacksburg_zips: return 'blacksburg'
        if zip_code in christiansburg_zips: return 'christiansburg'
        if zip_code in roanoke_lynchburg_zips: return 'roanoke_lynchburg'
        if zip_code in dc_zips: return 'DC'
        if zip_code in ny_zips: return 'NY'
        if zip_code in charlotte_zips: return 'Charlotte'
        return 'Unknown'

    df_dentists_with_locs['location_name'] = df_dentists_with_locs['zip'].apply(map_zip_to_location)
    df_regional_dentists = df_dentists_with_locs[df_dentists_with_locs['location_name'] != 'Unknown'].copy()
    print(f"Filtered to {len(df_regional_dentists)} records within the target regions.")

    # Show missing zipcode compare NPI to map API
    print("\nChecking for presence of specified ZIP codes in the filtered NPI data:")
    all_target_zips = set(blacksburg_zips + christiansburg_zips + roanoke_lynchburg_zips + dc_zips + ny_zips + charlotte_zips)
    present_zips = df_regional_dentists['zip'].unique()
    missing_zips = all_target_zips - set(present_zips)

    if missing_zips:
        print(f"  Warning: The following specified ZIP codes were not found in the filtered NPI data: {missing_zips}")
    else:
        print("  All specified ZIP codes are present in the filtered NPI data.")

    # Display count of records per region
    print("\nCount of NPI records per region:")
    display(df_regional_dentists['location_name'].value_counts())


    df_regional_dentists['name'] = df_regional_dentists['Provider Organization Name (Legal Business Name)']
    is_individual = df_regional_dentists['Entity Type Code'] == 1
    df_regional_dentists.loc[is_individual, 'name'] = df_regional_dentists['Provider First Name'].str.cat(df_regional_dentists['Provider Last Name (Legal Name)'], sep=' ', na_rep='').str.strip()

    output_cols = ['NPI', 'name', 'Entity Type Code', 'address_1', 'address_2', 'zip', 'location_name']
    final_df = df_regional_dentists[output_cols]

    os.makedirs(temp_output_path, exist_ok=True)
    preprocessed_filename = os.path.join(temp_output_path, "dentists_in_regions.csv")
    final_df.to_csv(preprocessed_filename, index=False, encoding='utf-8-sig')

    print(f"\nPre-processing complete. The file 'dentists_in_regions.csv' has been saved to: {preprocessed_filename}")

    print("\nPreview of the final pre-processed dentist data:")
    display(final_df.head())
else:
    print("\nNo dentist records found in the main NPI data file. Processing stopped.")

In [ ]:
# Merge with previous map api result
base_path = "/content/drive/MyDrive/RA/health_care/dentist_all_keywords"
final_output_path = os.path.join(base_path, "output", "final")
temp_path = os.path.join(base_path, "temp")
final_processed_path = os.path.join(final_output_path, "final_processed.csv")
dentists_in_regions_path = os.path.join(temp_path, "dentists_in_regions.csv")
merged_output_file = os.path.join(final_output_path, "final_data_with_npi.csv")


# Load the Datasets
print("Loading final_processed.csv file...")
try:
    df_final = pd.read_csv(final_processed_path)
    print(f"Successfully loaded {len(df_final)} records from final_processed.csv.")
except FileNotFoundError:
    print(f"Error: The file '{final_processed_path}' was not found.")
    df_final = pd.DataFrame()

print("Loading dentists_in_regions.csv file...")
try:
    df_npi = pd.read_csv(dentists_in_regions_path)
    print(f"Successfully loaded {len(df_npi)} records from dentists_in_regions.csv.")
except FileNotFoundError:
    print(f"Error: The file '{dentists_in_regions_path}' was not found.")
    df_npi = pd.DataFrame()

# Merge
if not df_final.empty and not df_npi.empty:
    print("\nPreparing data for merging...")

    df_final['title_normalized'] = df_final['title'].str.lower().str.replace(r'[^\w\s]', '', regex=True)
    df_npi['name_normalized'] = df_npi['name'].str.lower().str.replace(r'[^\w\s]', '', regex=True)

    df_final['zip'] = pd.to_numeric(df_final['zip'], errors='coerce').astype('Int64').astype(str).replace('<NA>', '')
    df_npi['zip'] = pd.to_numeric(df_npi['zip'], errors='coerce').astype('Int64').astype(str).replace('<NA>', '')

    # Aggregate
    print("Aggregating NPI data...")

    agg_funcs = {
        'NPI': lambda x: ', '.join(x.astype(str).unique()),
        'name': 'first',
        'Entity Type Code': 'first',
        'address_1': 'first',
        'address_2': 'first'
    }

    df_npi_aggregated = df_npi.groupby(['name_normalized', 'zip']).agg(agg_funcs).reset_index()

    df_npi_to_merge = df_npi_aggregated.rename(columns={
        'name': 'NPI_Name',
        'NPI': 'NPI_Matched',
        'Entity Type Code': 'NPI_EntityType',
        'address_1': 'NPI_Address_1',
        'address_2': 'NPI_Address_2'
    })

    print("Performing a left merge...")

    df_merged = pd.merge(
        df_final,
        df_npi_to_merge,
        left_on=['title_normalized', 'zip'],
        right_on=['name_normalized', 'zip'],
        how='left'
    )

    df_merged.drop(columns=['title_normalized', 'name_normalized'], inplace=True)

    df_merged.to_csv(merged_output_file, index=False, encoding='utf-8-sig')


    matched_count = df_merged['NPI_Matched'].notna().sum()
    total_count = len(df_merged)

    print("\n--- Merge Complete ---")
    print(f"The final merged file has been saved to: {merged_output_file}")
    print(f"Matched {matched_count} out of {total_count} records with an NPI number.")

    print("\nPreview of the final merged data (with NPI columns):")
    display(df_merged[df_merged['NPI_Matched'].notna()])

else:
    print("\nOne of the required files could not be loaded. Merging cannot proceed.")

# NPI records as Input

In [ ]:
# Only organizations
import pandas as pd
import os

nppes_data_path = "/content/drive/MyDrive/RA/health_care/NPI"
main_npi_file = "npidata_pfile_20050523-20251012.csv"

# ZIP codes for each region
blacksburg_zips = [24060, 24061, 24062, 24063]
christiansburg_zips = [24068, 24073]
roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA ZIPs
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
]

all_target_zips = blacksburg_zips + christiansburg_zips + roanoke_lynchburg_zips + dc_zips + charlotte_zips + ny_zips

exclusion_taxonomy_codes = [
    # Suppliers (eg: Medical Equipment)
    '332B00000X', '332S00000X', '333600000X', '335E00000X', '335U00000X',
    # Transportation Services
    '341600000X', '343900000X', '347B00000X', '347C00000X',
    # Managed Care / Insurance
    '302F00000X', '305S00000X',
    # Government Agencies
    '251K00000X', '251S00000X', '251T00000X', '251V00000X',
    # Other Service Providers(eg: social works)
    '174400000X',
    # Schools
    '352Y00000X'
]

main_columns_to_load = [
    'Entity Type Code',
    'Provider Organization Name (Legal Business Name)',
    'Provider Business Practice Location Address Postal Code'
]+ [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]

organization_chunks = []
print("Processing main NPI file in chunks to find organizations in target ZIPs...")

try:
    # Chunk df
    chunk_iterator = pd.read_csv(
        os.path.join(nppes_data_path, main_npi_file),
        usecols=main_columns_to_load,
        chunksize=100000,
        low_memory=False,
        encoding='utf-8'
    )

    for i, chunk in enumerate(chunk_iterator):
        print(f"  - Processing chunk {i+1}...")

        orgs_in_chunk = chunk[chunk['Entity Type Code'] == 2].copy()

        if not orgs_in_chunk.empty:
            # Create a mask to exclude unwanted organizations
            taxonomy_cols = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]
            exclusion_mask = orgs_in_chunk[taxonomy_cols].isin(exclusion_taxonomy_codes).any(axis=1)

            filtered_orgs = orgs_in_chunk[~exclusion_mask].copy()

            if not filtered_orgs.empty:
                filtered_orgs['zip'] = filtered_orgs['Provider Business Practice Location Address Postal Code'].astype(str).str.split('-').str[0]
                filtered_orgs.dropna(subset=['zip'], inplace=True)
                filtered_orgs['zip'] = pd.to_numeric(filtered_orgs['zip'], errors='coerce')
                filtered_orgs.dropna(subset=['zip'], inplace=True)
                filtered_orgs['zip'] = filtered_orgs['zip'].astype(int)

                regional_orgs = filtered_orgs[filtered_orgs['zip'].isin(all_target_zips)]

                if not regional_orgs.empty:
                    organization_chunks.append(regional_orgs)

    if organization_chunks:
        df_orgs_regional = pd.concat(organization_chunks, ignore_index=True)
        def map_zip_to_location(zip_code):
            if zip_code in blacksburg_zips: return 'blacksburg'
            if zip_code in christiansburg_zips: return 'christiansburg'
            if zip_code in roanoke_lynchburg_zips: return 'roanoke_lynchburg'
            if zip_code in dc_zips: return 'washington_dc'
            if zip_code in charlotte_zips: return 'charlotte_nc'
            if zip_code in ny_zips: return 'new_york_ny'
            return 'Unknown'

        df_orgs_regional['location_name'] = df_orgs_regional['zip'].apply(map_zip_to_location)
        df_npi_keywords = df_orgs_regional[['Provider Organization Name (Legal Business Name)', 'location_name']].copy()
        df_npi_keywords.rename(columns={'Provider Organization Name (Legal Business Name)': 'keyword'}, inplace=True)
        df_npi_keywords.dropna(subset=['keyword'], inplace=True)
        df_npi_keywords.drop_duplicates(inplace=True)
        print(f"\nProcessing complete. Extracted {len(df_npi_keywords)} unique organization-location pairs to use as keywords.")

    else:
        print("No regional organizations found.")
        df_npi_keywords = pd.DataFrame()

except FileNotFoundError as e:
    print(f"\nError: Could not find the main NPI file: {e}")
    df_npi_keywords = pd.DataFrame()

In [ ]:
# Organizations and individual providers
import pandas as pd
import os

nppes_data_path = "/content/drive/MyDrive/RA/health_care/NPI"
main_npi_file = "npidata_pfile_20050523-20251012.csv"

# ZIP codes for each region
blacksburg_zips = [24060, 24061, 24062, 24063]
christiansburg_zips = [24068, 24073]
roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA ZIPs
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
]
all_target_zips = blacksburg_zips + christiansburg_zips + roanoke_lynchburg_zips + dc_zips + charlotte_zips + ny_zips

main_columns_to_load = [
    'NPI', 'Entity Type Code',
    'Provider Organization Name (Legal Business Name)',
    'Provider Last Name (Legal Name)',
    'Provider First Name',
    'Provider Business Practice Location Address Postal Code'
]

provider_chunks = []
print("Processing main NPI file in chunks to find all providers in target ZIPs...")

try:
    chunk_iterator = pd.read_csv(
        os.path.join(nppes_data_path, main_npi_file),
        usecols=main_columns_to_load,
        chunksize=100000,
        low_memory=False,
        encoding='utf-8'
    )

    for i, chunk in enumerate(chunk_iterator):
        print(f"  - Processing chunk {i+1}...")

        provider_chunk = chunk.copy()

        # Clean ZIP codes and filter by target regions
        provider_chunk['zip'] = provider_chunk['Provider Business Practice Location Address Postal Code'].astype(str).str.split('-').str[0]
        provider_chunk.dropna(subset=['zip'], inplace=True)
        provider_chunk['zip'] = pd.to_numeric(provider_chunk['zip'], errors='coerce')
        provider_chunk.dropna(subset=['zip'], inplace=True)
        provider_chunk['zip'] = provider_chunk['zip'].astype(int)

        regional_providers = provider_chunk[provider_chunk['zip'].isin(all_target_zips)]

        if not regional_providers.empty:
            provider_chunks.append(regional_providers)

    if provider_chunks:
        df_providers_regional = pd.concat(provider_chunks, ignore_index=True)

        def map_zip_to_location(zip_code):
            if zip_code in blacksburg_zips: return 'blacksburg'
            if zip_code in christiansburg_zips: return 'christiansburg'
            if zip_code in roanoke_lynchburg_zips: return 'roanoke_lynchburg'
            if zip_code in dc_zips: return 'washington_dc'
            if zip_code in charlotte_zips: return 'charlotte_nc'
            if zip_code in ny_zips: return 'new_york_ny'
            return 'Unknown'

        df_providers_regional['location_name'] = df_providers_regional['zip'].apply(map_zip_to_location)

        # For organizations, use the organization name
        df_providers_regional['keyword'] = df_providers_regional['Provider Organization Name (Legal Business Name)']

        # For individual providers, create the full name
        is_individual = df_providers_regional['Entity Type Code'] == 1
        full_name = df_providers_regional.loc[is_individual, 'Provider First Name'].str.cat(
            df_providers_regional.loc[is_individual, 'Provider Last Name (Legal Name)'],
            sep=' ',
            na_rep=''
        ).str.strip()

        df_providers_regional.loc[is_individual, 'keyword'] = full_name


        df_npi_keywords = df_providers_regional[['keyword', 'location_name']].copy()
        df_npi_keywords.dropna(subset=['keyword'], inplace=True)
        df_npi_keywords = df_npi_keywords[df_npi_keywords['keyword'] != '']
        df_npi_keywords.drop_duplicates(inplace=True)

        print(f"\nProcessing complete. Extracted {len(df_npi_keywords)} unique provider-location pairs to use as keywords.")
    else:
        print("No regional providers found.")
        df_npi_keywords = pd.DataFrame()

except FileNotFoundError as e:
    print(f"\nError: Could not find the main NPI file: {e}")
    df_npi_keywords = pd.DataFrame()

In [ ]:
# Post NPI Keywords to DataForSEO Maps API
print("--- Starting Task POST for NPI Keywords ---")

if df_npi_keywords.empty:
    print("Keyword DataFrame is empty. Stopping task posting.")
else:
    npi_search_temp_dir = os.path.join(base_path, "temp", "npi_map_search")
    os.makedirs(npi_search_temp_dir, exist_ok=True)
    task_list_csv_path = os.path.join(npi_search_temp_dir, "task_list.csv")

    # Load existing tasks to skip
    existing_tasks = set()
    try:
        if os.path.exists(task_list_csv_path):
            tasks_df = pd.read_csv(task_list_csv_path)
            if not tasks_df.empty:
                for index, row in tasks_df.iterrows():
                    existing_tasks.add((row['location_name'], row['keyword']))
            print(f"Loaded {len(existing_tasks)} existing tasks to skip.")
    except pd.errors.EmptyDataError:
        print("Task list file is empty. Starting fresh.")

    locations_to_search = {
        'roanoke_lynchburg': 200573,
        'blacksburg': 1027041,
        'christiansburg': 1027077,
        'washington_dc': 200511,
        'charlotte_nc': 200517,
        'new_york_ny' : 21167
    }
    # 'location_name': ['roanoke_lynchburg', 'blacksburg', 'christiansburg','DC','NY','Charlotte',None,None,None,None,None],
    # 'location_code': [200573, 1027041, 1027077,200511,21167,200517,None,None,None,None,None],

    maps_api_url = "https://api.dataforseo.com/v3/serp/google/maps/task_post"
    write_header = not os.path.exists(task_list_csv_path) or (os.path.getsize(task_list_csv_path) == 0)

    # Group keywords by their location
    grouped_keywords = df_npi_keywords.groupby('location_name')

    with open(task_list_csv_path, 'a', newline='', encoding='utf-8-sig') as f:
        import csv
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["task_id", "api_type", "location_name", "keyword", "raw_json_path"])

        # Iterate through each location group
        for loc_name, group in grouped_keywords:
            if loc_name not in locations_to_search:
                print(f"\nSkipping location '{loc_name}' as it has no corresponding location code.")
                continue

            loc_code = locations_to_search[loc_name]
            print(f"\nProcessing tasks for location: '{loc_name}' using code {loc_code}")

            # Get keywords for this specific location
            keywords_for_location = group['keyword'].tolist()

            for keyword in keywords_for_location:
                # Skip task already exists
                if (loc_name, keyword) in existing_tasks:
                    print(f"  Skipping already posted task for keyword: '{keyword}'")
                    continue

                task_id = post_dataforseo_task(maps_api_url, loc_code, keyword)

                if task_id:
                    safe_keyword = "".join(c for c in keyword if c.isalnum() or c in (' ', '_')).rstrip().replace(' ', '_')
                    raw_json_filename = f"{loc_name}_{safe_keyword[:100]}_maps.json" # Truncate long filenames
                    raw_json_path = os.path.join(npi_search_temp_dir, raw_json_filename)

                    writer.writerow([task_id, "maps", loc_name, keyword, raw_json_path])
                    f.flush()

                time.sleep(1)

    print(f"\nAll targeted NPI keyword tasks posted.")
    print("!!! IMPORTANT: Please wait 20 minutes before running the next step. !!!")

In [ ]:
base_path = "/content/drive/MyDrive/RA/health_care/dentist_all_keywords"
npi_search_temp_dir = os.path.join(base_path, "temp", "npi_map_search")
task_list_csv_path = os.path.join(npi_search_temp_dir, "task_list.csv")
final_output_dir = os.path.join(base_path, "output", "final")
final_csv_path = os.path.join(final_output_dir, "npi_map_search_final.csv")
os.makedirs(final_output_dir, exist_ok=True)

tasks_to_get = []
try:
    tasks_df = pd.read_csv(task_list_csv_path)
    tasks_to_get = tasks_df.to_dict('records')
    print(f"Found {len(tasks_to_get)} tasks in the task list.")

    for task in tasks_to_get:
        # Skip if the JSON file already exists
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping GET: Result file already exists for Keyword='{task['keyword']}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll result retrieval attempts completed.")

except FileNotFoundError:
    print(f"Error: Task list file not found at '{task_list_csv_path}'. Cannot get results.")


if tasks_to_get:
    json_files = glob.glob(os.path.join(npi_search_temp_dir, "*.json"))
    npi_maps_data = []

    print(f"\nFound {len(json_files)} JSON files to process.")
    for file in json_files:
        npi_maps_data.extend(parse_maps_results(file))

    if npi_maps_data:
        df_processed = pd.DataFrame(npi_maps_data)
        print(f"Parsed {len(df_processed)} total records from all JSON files.")

        # Deduplicate
        df_deduplicated = df_processed.drop_duplicates(subset=['title', 'address'], keep='first').copy()
        print(f"After deduplication, {len(df_deduplicated)} unique records remain.")

        # Sort by rating and votes
        df_sorted = df_deduplicated.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        final_columns = [
            'title', 'address', 'latitude', 'longitude', 'zip',
            'rating_value', 'votes_count', 'rating_1_star', 'rating_2_star',
            'rating_3_star', 'rating_4_star', 'rating_5_star', 'type'
        ]
        for col in final_columns:
            if col not in df_sorted.columns:
                df_sorted[col] = None

        df_final_output = df_sorted[final_columns]

        df_final_output.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nProcessing complete. Final data saved to: {final_csv_path}")

        print("\nPreview of the final processed NPI Map Search data:")
        display(df_final_output.head())
    else:
        print("\nNo data was extracted from the JSON result files.")
else:
    print("\nNo tasks were found in the task list, so no data to process.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_all_keywords/output/final/npi_map_search_final.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully npi_map_search_final.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    # ZIP codes for each region
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA) ZIP
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    # map ZIP code to location
    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'

        if zip_code in blacksburg_zips:
            return 'Blacksburg'
        elif zip_code in christiansburg_zips:
            return 'Christiansburg'
        elif zip_code in roanoke_lynchburg_zips:
            return 'Roanoke/Lynchburg'
        elif zip_code in dc_zips:
            return 'DC'
        elif zip_code in ny_zips:
            return 'NY'
        elif zip_code in charlotte_zips:
            return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)

    # Filter out locations don't belong to any of the defined regions
    df_filtered = df[df['location_name'] != 'Unknown'].copy()


    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value'], inplace=True)

    print(f"After mapping ZIP codes, {len(df_filtered)} records remain for analysis.")


    # Boxplot
    plt.figure(figsize=(12, 7))
    sns.boxplot(x='location_name', y='rating_value', data=df_filtered)
    plt.title('Distribution of Ratings by zipcode')
    plt.xlabel('Region')
    plt.ylabel('Rating Value')
    plt.grid(True)
    plt.show()

    # Histograms
    plt.figure(figsize=(14, 8))
    sns.histplot(data=df_filtered, x='rating_value', hue='location_name', multiple='dodge', shrink=.8, bins=15)
    plt.title('Histogram of Ratings by zipcode')
    plt.xlabel('Rating Value')
    plt.ylabel('Count')
    plt.grid(True)
    plt.show()

    # Violin Plot of Rating Value
    plt.figure(figsize=(12, 7))
    sns.violinplot(x='location_name', y='rating_value', data=df_filtered)
    plt.title('Distribution of Rating Value by zipcode')
    plt.xlabel('Region')
    plt.ylabel('Rating Value')
    plt.grid(True)
    plt.show()


    rating_summary = df_filtered.groupby('location_name')['rating_value'].describe()
    print("\nSummary Statistics for Rating Value by zipcode")
    print(rating_summary)

    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    star_cols_exist = all(col in df_filtered.columns for col in star_cols)

    if star_cols_exist:
        for col in star_cols:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)

        star_distribution = df_filtered.groupby('location_name')[star_cols].mean()
        print("\nStar Rating Distribution (Average Votes per Star per Region):")
        display(star_distribution)

        # Stacked Bar Chart
        star_distribution.plot(kind='bar', stacked=True, figsize=(14, 8))
        plt.title('Average Star Rating Distribution by ZIP Code-Defined Region')
        plt.xlabel('Region')
        plt.ylabel('Average Number of Votes')
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Star Rating')
        plt.tight_layout()
        plt.show()
    else:
        print("\nStar rating columns not found. Skipping star rating analysis.")